## Model Development and Fine Tuning ## 

**Importing Libraries** 

In [41]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import brier_score_loss
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV

**Loading The Data** 

In [28]:
df = pd.read_csv("C:\\Users\\Admin\\Downloads\\pitch_data_FeatureEngineered(3).csv")

**Pre-Processing the Data** 

In [29]:

# Shuffle the data to randomize the order
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

## Model Development ## 

**Feature Sets**

In [37]:

feature_sets = {
    'core': ['plate_location_x', 'plate_location_z'],
    'pitch_features': ['plate_location_x', 'plate_location_z', 'rel_speed', 'spin_rate', 'induced_vert_break', 'horizontal_break'],
    'smoothed_rates': ['plate_location_x', 'plate_location_z', 'smoothed_pitcher_rate', 'smoothed_umpire_rate', 'smoothed_pair_rateup', 'smoothed_pair_ratecu','smoothed_pair_rateub'],
    'All Features': ['plate_location_x', 'plate_location_z', 'rel_speed', 'spin_rate', 'induced_vert_break', 'horizontal_break', 
                     'smoothed_pitcher_rate', 'smoothed_umpire_rate', 'smoothed_pair_rateup', 'smoothed_pair_ratecu','smoothed_pair_rateub',
                     'pitch_type_CB','pitch_type_CH','pitch_type_FC','pitch_type_FF','pitch_type_FS','pitch_type_FT','pitch_type_OT','pitch_type_SL',
                     'inning_1','inning_2','inning_3','inning_4','inning_5','inning_6','inning_7','inning_8','inning_9']
}


**Defining Models** 

Due to the fact that the model will have to predict probabilites for each pitch the models that will be tested are Logistic Regression, Random Forest and Gradient Boosting 

In [32]:
# Define models and hyperparameters manually
models = {
    'Logistic Regression': {
        'model': LogisticRegression(),
        'params': [
            {'C': 0.1, 'penalty': 'l2'},
            {'C': 1, 'penalty': 'l2'},
            {'C': 10, 'penalty': 'l2'}
        ]
    },
    'Random Forest': {
        'model': RandomForestClassifier(),
        'params': [
            {'n_estimators': 100, 'max_depth': None},
            {'n_estimators': 100, 'max_depth': 10}
        ]
    },
    'Gradient Boosting': {
        'model': GradientBoostingClassifier(),
        'params': [
            {'learning_rate': 0.01, 'n_estimators': 100, 'max_depth': 3},
            {'learning_rate': 0.1, 'n_estimators': 100, 'max_depth': 5}
        ]
    }
}


**Parameter Tuning for Each Model and Pipeline Creation**

**Iterate Over Feature Sets**:
   - The code loops through each feature set defined in `feature_sets`, which specifies different subsets of features to be used for model training.

 **Iterate Over Models and Parameter Sets**:
   - For each feature set, the code loops over all specified models in `models`.
   - Each model has a set of hyperparameters to be tested, which are defined in the `params` attribute of each `model_info` entry.

**Pipeline Creation and Model Training**:
   - A pipeline is created for each parameter combination, which includes:
     - **Scaling**: A `StandardScaler` is applied to standardize the features.
     - **Model Training**: The model is configured with the current parameter set using `.set_params(**params)` and is then trained on the full dataset.
   - The trained pipeline is used to predict probabilities (`y_pred_prob`) for the entire dataset.

**Calculate Brier Score**:
   - The Brier score, which measures the accuracy of probability predictions, is calculated for each parameter combination.
   - The Brier score reflects the mean squared difference between predicted probabilities and actual binary outcomes, with lower scores indicating better calibration and accuracy.

**Identify and Save Incorrect High-Confidence Predictions**:
   - Predictions with high confidence but incorrect outcomes are identified:
     - Confidently predicted strikes (≥ 0.5 probability) that were not strikes.
     - Confidently predicted balls (< 0.5 probability) that were actually strikes.


In [33]:
final_results = []
# Iterate over feature sets and models
for feature_set_name, features in feature_sets.items():
    X = df[features]
    y = df['is_strike']

    for model_name, model_info in models.items():
        best_model = None
        best_brier_score = float('inf')

        for params in model_info['params']:
            # Create pipeline and fit on the entire dataset
            pipeline = Pipeline([
                ('scaler', StandardScaler()),
                ('model', model_info['model'].set_params(**params))
            ])
            pipeline.fit(X, y)

            # Predict probabilities for the entire dataset
            y_pred_prob = pipeline.predict_proba(X)[:, 1]

            # Calculate Brier score for the entire dataset
            brier_score = brier_score_loss(y, y_pred_prob)

            # Add the predicted probabilities back to the data
            df['predicted_prob_strike'] = y_pred_prob

            # Identify incorrect high-confidence predictions
            incorrect_predictions = df[
                ((df['predicted_prob_strike'] >= 0.5) & (df['is_strike'] == 0)) |
                ((df['predicted_prob_strike'] < 0.5) & (df['is_strike'] == 1))
            ]

            # Count of incorrect high-confidence predictions
            incorrect_count = incorrect_predictions.shape[0]

            # Store the results for the current feature set and model combination
            final_results.append({
                'feature_set': feature_set_name,
                'model': model_name,
                'overall_brier_score': brier_score,
                'incorrect_high_confidence_count': incorrect_count,
                'best_params': params
            })

            print(f"Model: {model_name} | Feature Set: {feature_set_name} | Brier Score: {brier_score} | Incorrect High-Confidence Count: {incorrect_count} | Params: {params}")

            # Save incorrect predictions for further analysis
            incorrect_predictions.to_csv(f"incorrect_high_confidence_predictions_{model_name}_{feature_set_name}.csv", index=False)

            # Remove the predicted probabilities column for the next model run
            df = df.drop(columns=['predicted_prob_strike'])

Model: Logistic Regression | Feature Set: core | Brier Score: 0.21437845654546578 | Incorrect High-Confidence Count: 17428 | Params: {'C': 0.1, 'penalty': 'l2'}
Model: Logistic Regression | Feature Set: core | Brier Score: 0.21437924923444693 | Incorrect High-Confidence Count: 17428 | Params: {'C': 1, 'penalty': 'l2'}
Model: Logistic Regression | Feature Set: core | Brier Score: 0.21437932866798623 | Incorrect High-Confidence Count: 17428 | Params: {'C': 10, 'penalty': 'l2'}
Model: Random Forest | Feature Set: core | Brier Score: 0.00855822940046114 | Incorrect High-Confidence Count: 18 | Params: {'n_estimators': 100, 'max_depth': None}
Model: Random Forest | Feature Set: core | Brier Score: 0.04670477138139318 | Incorrect High-Confidence Count: 3777 | Params: {'n_estimators': 100, 'max_depth': 10}
Model: Gradient Boosting | Feature Set: core | Brier Score: 0.10542540270019793 | Incorrect High-Confidence Count: 4366 | Params: {'learning_rate': 0.01, 'n_estimators': 100, 'max_depth': 3}

**Saving and Presenting Final Results**

In [34]:
# Convert final results to DataFrame and display
final_results_df = pd.DataFrame(final_results)
print(final_results_df.sort_values(by='overall_brier_score', ascending=True))


       feature_set                model  overall_brier_score  \
10  pitch_features        Random Forest             0.007439   
17  smoothed_rates        Random Forest             0.007690   
24    All Features        Random Forest             0.007986   
3             core        Random Forest             0.008558   
27    All Features    Gradient Boosting             0.045037   
13  pitch_features    Gradient Boosting             0.045638   
4             core        Random Forest             0.046705   
20  smoothed_rates    Gradient Boosting             0.046802   
6             core    Gradient Boosting             0.047780   
11  pitch_features        Random Forest             0.049852   
18  smoothed_rates        Random Forest             0.051360   
25    All Features        Random Forest             0.076766   
19  smoothed_rates    Gradient Boosting             0.105425   
12  pitch_features    Gradient Boosting             0.105425   
26    All Features    Gradient Boosting 

## Comparing Gradient Boosting to Random Forest To Check for Overfitting and Performance on Validation Set 

In [38]:


# Define specific parameters for each model
model_params = {
    ('Random Forest', 'pitch_features'): {'n_estimators': 100, 'max_depth': None},
    ('Random Forest', 'smoothed_rates'): {'n_estimators': 100, 'max_depth': None},
    ('Random Forest', 'All Features'): {'n_estimators': 100, 'max_depth': None},
    ('Random Forest', 'core'): {'n_estimators': 100, 'max_depth': None},
    ('Gradient Boosting', 'All Features'): {'n_estimators': 100, 'max_depth': 5, 'learning_rate': 0.1},
    ('Gradient Boosting', 'pitch_features'): {'n_estimators': 100, 'max_depth': 5, 'learning_rate': 0.1}
}

# Define models
models = {
    'Random Forest': RandomForestClassifier(random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(random_state=42)
}

# Initialize Stratified K-Fold for cross-validation
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Prepare to store cross-validation results
results = []

for feature_set_name, features in feature_sets.items():
    X = df[features]
    y = df['is_strike']
    
    for model_name, model in models.items():
        # Check if model and feature set have specific parameters
        if (model_name, feature_set_name) in model_params:
            params = model_params[(model_name, feature_set_name)]
            model.set_params(**params)
            
            train_brier_scores = []
            val_brier_scores = []
            feature_importances = []

            # Perform Stratified K-Fold Cross-validation
            for train_index, val_index in skf.split(X, y):
                X_train, X_val = X.iloc[train_index], X.iloc[val_index]
                y_train, y_val = y.iloc[train_index], y.iloc[val_index]

                # Create pipeline
                pipeline = Pipeline([
                    ('scaler', StandardScaler()),
                    ('model', model)
                ])
                
                # Train model
                pipeline.fit(X_train, y_train)
                
                # Calculate Brier scores for training and validation sets
                y_train_pred_prob = pipeline.predict_proba(X_train)[:, 1]
                y_val_pred_prob = pipeline.predict_proba(X_val)[:, 1]

                train_brier_score = brier_score_loss(y_train, y_train_pred_prob)
                val_brier_score = brier_score_loss(y_val, y_val_pred_prob)

                train_brier_scores.append(train_brier_score)
                val_brier_scores.append(val_brier_score)
                
                # Extract feature importances if the model has them
                if hasattr(model, 'feature_importances_'):
                    feature_importances.append(model.feature_importances_)
            
            # Average feature importances across folds
            avg_feature_importances = np.mean(feature_importances, axis=0) if feature_importances else None

            # Store results
            results.append({
                'feature_set': feature_set_name,
                'model': model_name,
                'params': params,
                'avg_train_brier_score': np.mean(train_brier_scores),
                'avg_val_brier_score': np.mean(val_brier_scores),
                'feature_importances': avg_feature_importances
            })

            print(f"Model: {model_name} | Feature Set: {feature_set_name} | Params: {params} | Avg Train Brier: {np.mean(train_brier_scores)} | Avg Val Brier: {np.mean(val_brier_scores)}")

# Convert results to DataFrame for easier analysis
results_df = pd.DataFrame(results)

# Display the results sorted by average validation Brier score
print(results_df.sort_values(by='avg_val_brier_score', ascending=True))

Model: Random Forest | Feature Set: core | Params: {'n_estimators': 100, 'max_depth': None} | Avg Train Brier: 0.00854130080662833 | Avg Val Brier: 0.060133210950888916
Model: Random Forest | Feature Set: pitch_features | Params: {'n_estimators': 100, 'max_depth': None} | Avg Train Brier: 0.007488831489973924 | Avg Val Brier: 0.05324253394478914
Model: Gradient Boosting | Feature Set: pitch_features | Params: {'n_estimators': 100, 'max_depth': 5, 'learning_rate': 0.1} | Avg Train Brier: 0.04478120431644404 | Avg Val Brier: 0.051984132899059676
Model: Random Forest | Feature Set: smoothed_rates | Params: {'n_estimators': 100, 'max_depth': None} | Avg Train Brier: 0.007736858196205376 | Avg Val Brier: 0.054853694811617656
Model: Random Forest | Feature Set: All Features | Params: {'n_estimators': 100, 'max_depth': None} | Avg Train Brier: 0.008364320205017536 | Avg Val Brier: 0.05905733297365344
Model: Gradient Boosting | Feature Set: All Features | Params: {'n_estimators': 100, 'max_dep

## Model Selection 

### Why Choose Gradient Boosting with All Features?

In this analysis, several models were tested using different feature sets, and the results indicate that the **Gradient Boosting model with All Features** is the most effective choice. Here are the key reasons:
**Lowest Validation Brier Score**:
   - The Gradient Boosting model with all features achieved the lowest validation Brier score, indicating the highest accuracy in probability predictions.
   - A lower Brier score means that the model provides well-calibrated probability estimates, making it more reliable for predicting whether a pitch is a strike.

 **Enhanced Generalization and Reduced Overfitting**:
   - Although Random Forest is generally effective at reducing overfitting, the results show that the Gradient Boosting model provides better generalization with this dataset.
   - By leveraging all available features, Gradient Boosting balances variance and bias more effectively, producing a model that is both accurate and reliable.
   - There is a large descrepancy between Random Forest Models training brier score and validation brier score suggesting overfitting on the training data 

Overall, the Gradient Boosting model with All Features demonstrates superior performance in accuracy and feature interaction modeling. Its ability to handle diverse features and provide accurate probability estimates makes it the optimal choice for this task.

* Note - Features with a feature importance less then.005 were removed this included 15 features with the final list of features for the final model = [
    'plate_location_x', 'plate_location_z', 'rel_speed', 'spin_rate', 'induced_vert_break', 'horizontal_break', 
    'smoothed_pitcher_rate', 'smoothed_umpire_rate', 'smoothed_pair_rateup', 'smoothed_pair_ratecu', 'smoothed_pair_rateub',
    'pitch_type_CB', 'pitch_type_CH', 'pitch_type_FC', 'pitch_type_FF', 'pitch_type_FS'
]



## Further Parameter Tuning on Gradient Boosting Model 

In [43]:

X = df[feature_sets['All Features']]
y = df['is_strike']
# Define the Gr]adient Boosting model
gb_model = GradientBoostingClassifier(random_state=42)

# Set up the parameter grid for tuning
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.05, 0.1],
    'subsample': [0.8, 1.0]
}

# Initialize Grid Search with cross-validation
grid_search = GridSearchCV(
    estimator=gb_model,
    param_grid=param_grid,
    scoring='neg_brier_score',  # Use negative Brier score for optimization
    cv=5,  # 5-fold cross-validation
    verbose=1,
    n_jobs=-1  # Use all available cores for parallel processing
)

# Perform the grid search to find the best parameters
grid_search.fit(X, y)

# Output the best parameters and corresponding Brier score
best_params = grid_search.best_params_
best_score = -grid_search.best_score_  # Convert back from negative score

print("Best Parameters:", best_params)
print("Best Brier Score:", best_score)


Fitting 5 folds for each of 54 candidates, totalling 270 fits
Best Parameters: {'learning_rate': 0.05, 'max_depth': 3, 'n_estimators': 300, 'subsample': 1.0}
Best Brier Score: 0.05142542657220213


In [44]:
import pandas as pd
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import brier_score_loss
#Import the Pre Processed Data Generated at the End of the Feature Engineering Notebook 
df = pd.read_csv("C:\\Users\\Admin\\Downloads\\pitch_data_FeatureEngineered(3).csv")

# Shuffle the data to randomize the order
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

# Updated feature set after removing low-importance features
# Updated feature set after removing low-importance features
updated_features = [
    'plate_location_x', 'plate_location_z', 'rel_speed', 'spin_rate', 'induced_vert_break', 'horizontal_break', 
    'smoothed_pitcher_rate', 'smoothed_umpire_rate', 'smoothed_pair_rateup', 'smoothed_pair_ratecu', 'smoothed_pair_rateub',
    'pitch_type_CB', 'pitch_type_CH', 'pitch_type_FC', 'pitch_type_FF', 'pitch_type_FS'
]

X = df[updated_features]
y = df['is_strike']

# Define the Gradient Boosting model with the specific parameters
model = GradientBoostingClassifier(
    learning_rate=0.05,
    max_depth=3,
    n_estimators=300,
    subsample=1.0,
    random_state=42
)

# Create pipeline with scaling and the Gradient Boosting model
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', model)
])

# Train the model on the entire dataset
pipeline.fit(X, y)

# Calculate Brier score for the entire dataset
y_pred_prob = pipeline.predict_proba(X)[:, 1]
brier_score = brier_score_loss(y, y_pred_prob)

# Append the predictions for every pitch to the DataFrame
df['predicted_prob_strike'] = y_pred_prob

# Output the Brier score
print(f"Overall Brier Score: {brier_score}")

# Optionally save the updated DataFrame with predictions
df.to_csv("final_pitch_data_with_predictions.csv", index=False)

# Display the DataFrame with new prediction column for verification
print(df.head())
df

Overall Brier Score: 0.04890637974984191
   is_strike  is_swing  is_bottom  balls  strikes  outs_before  is_lhp  \
0          1         0          0      1        0            2       0   
1          0         0          1      1        0            2       1   
2          1         0          1      0        0            0       0   
3          0         0          1      0        1            1       1   
4          0         0          1      0        1            2       0   

   is_lhb  bat_score_before  field_score  ...  inning_10 inning_11 inning_12  \
0       1                 1            2  ...      False     False     False   
1       0                 1            8  ...      False     False     False   
2       1                 3            6  ...      False     False     False   
3       1                 0            0  ...      False     False     False   
4       0                 0            1  ...      False     False     False   

  inning_13 inning_14  inning_15 